> **说明**：本 notebook 是 [langchain-ai/langgraph-101](https://github.com/langchain-ai/langgraph-101) 的**简体中文翻译版**，供 Google Colab / Jupyter 使用。
>
> - 原文仓库：<https://github.com/langchain-ai/langgraph-101>
> - 建议学习顺序：`101_langchain_langgraph` → `102_middleware` → `deep_agents` → `deep_agents_extra`
> - **代码单元格保持英文原样**（标识符、字符串、prompt 均未改动）；仅翻译 Markdown 与 `#` 注释。
> - 图片路径仍相对原仓库；在 Colab 中请先 clone 原仓库或自行调整图片路径。


# LangGraph 101：构建你的第一个智能体（Agent）

<img src="../../images/react_agent.png" style="width: auto; max-height: 500px; border-radius: 8px;">

欢迎来到 LangGraph 101！本 notebook 将带你掌握用 LangChain 与 LangGraph 构建智能体的核心概念。

**你将学到：**
- 如何与语言模型交互
- 使用消息（messages）与多轮对话
- 添加工具（tools）以扩展 LLM 能力
- 构建能推理并行动的智能体
- 添加记忆（memory）以保持上下文
- 流式（streaming）输出以改善用户体验
<br> 
<br> 
---
<br> 

> **说明：** 本教程使用 LangChain v1（alpha），这是上手 LLM 开发最简便的方式。LangChain 智能体构建于 LangGraph 之上，开箱即用提供持久执行、流式输出、人机协同（HITL, human-in-the-loop）与持久化（persistence）。


## 第 0 部分：环境准备与安装

首先安装必要依赖并配置环境。


In [ ]:
# 安装所需依赖包
# 运行 uv sync 安装依赖，或执行：
# !pip install --pre -U langchain langchain-anthropic langchain-openai python-dotenv

### 初始化你的 LLM


In [ ]:
# 将项目根目录加入 Python 路径，以便从 utils 模块导入
import sys
from pathlib import Path
project_root = Path().resolve().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 从集中式 utils 模块导入 model
# 避免各 notebook 重复代码，并使用一致的模型配置
from utils.models import model

# 备选：若想在本处内联定义模型而非使用集中配置，取消下面注释：
# from dotenv import load_dotenv
# load_dotenv(dotenv_path="../.env", override=True)
# from langchain.chat_models import init_chat_model
# model = init_chat_model("openai:o3-mini")

# 说明：若使用其他供应商（Azure、Bedrock、Vertex AI），请修改 utils/models.py
# 切换 LLM 供应商的详细说明见 utils/models.py


# 在无 thread id 的简单示例中抑制 uuid v7 警告
import warnings
warnings.filterwarnings('ignore', message='LangSmith now uses UUID v7')

## 第 1 部分：第一次 LLM 调用

LangChain 提供**统一的模型接口**，可跨所有供应商使用。这意味着你可以在 OpenAI、Anthropic、Google 等之间轻松切换，而无需改动业务代码。

让我们先初始化一个聊天模型。


In [ ]:
# 发起第一次调用！
response = model.invoke("What is LangChain?")
response.pretty_print()

### 要点：
- `init_chat_model()` 提供面向任意 LLM 供应商的标准化接口
- `.invoke()` 发送消息并返回响应
- 无供应商锁定 —— 可轻松换模型！


## 第 2 部分：理解消息（Messages）

**消息（Messages）** 是 LangChain 中模型上下文的基本单元。它们表示模型的输入与输出，同时携带内容与元数据。

常见消息类型：
- **SystemMessage** —— 规定模型应如何表现的指令
- **HumanMessage** —— 用户输入
- **AIMessage** —— 模型回复
- **ToolMessage** —— 工具执行结果


In [ ]:
from langchain.messages import HumanMessage, SystemMessage, AIMessage

# 用不同消息类型创建一段对话
messages = [
    SystemMessage(content="You are a helpful AI assistant that explains technical concepts simply."),
    HumanMessage(content="What is an agent?"),
]

response = model.invoke(messages)
response.pretty_print()

### 多轮对话

消息让维护对话历史变得简单：


In [ ]:
# 继续对话
messages.append(response)  # Add AI response to history
messages.append(HumanMessage(content="Can you give me an example?"))

response = model.invoke(messages)
response.pretty_print()

### 要点：
- 消息表示对话历史
- SystemMessage 设定模型行为
- 通过向列表追加消息即可构建多轮对话


## 第 3 部分：添加工具 —— 扩展 LLM 能力

LLM 擅长语言，但无法访问外部数据或执行动作。**工具（Tools）** 用来扩展能力。你可以给 LLM 一组工具；当它需要时，会指定要调用哪个工具。你的职责是执行工具，并把结果再喂回 LLM，让它决定下一步。

只需写一个带清晰说明的 Python 函数即可创建工具。LangChain 的 `@tool` 装饰器会把函数信息格式化成 LLM 所需的形式。

让我们创建几个简单工具：


In [ ]:
import requests
import json

from langchain_core.tools import tool


# 基础硬编码工具
@tool
def search_movies(genre: str) -> str:
    """Search for movies by genre."""
    # 真实应用中这里会查询电影数据库
    movies = {
        "sci-fi": "Dune, Interstellar, Blade Runner 2049",
        "comedy": "The Grand Budapest Hotel, Superbad, Knives Out",
        "action": "Mad Max: Fury Road, John Wick, Mission Impossible"
    }
    return movies.get(genre.lower(), "No movies found for that genre")

# 更贴近实际、会调用 API 的工具
@tool
def get_weather(latitude: float, longitude: float) -> str:
    """Get current temperature in Fahrenheit and weather code for given coordinates.

    Args:
        latitude: Latitude coordinate
        longitude: Longitude coordinate

    Returns:
        JSON string with temperature_fahrenheit and weather_code (do not include the code in your response, translate it to plain English)
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,weather_code",
        "temperature_unit": "fahrenheit"
    }

    weather = requests.get(url, params=params).json()["current"]
    temperature = weather["temperature_2m"]
    weather_code = weather["weather_code"]
    result = {
        "temperature_fahrenheit": temperature,
        "weather_code": weather_code
    }

    return json.dumps(result)



# 用旧金山坐标直接测试工具
print(get_weather.invoke({"latitude": 37.77, "longitude": 122.42}))

### 工具调用（Tool Calling / Function Calling）

现在用 `.bind_tools()` 把这些工具交给模型：


In [ ]:
# 将工具绑定到模型
tools = [get_weather, search_movies]
model_with_tools = model.bind_tools(tools)
message = "What's the weather like in Seattle? (Seattle's coordinates are approximately 47.6° N latitude and 122.33° W longitude) "

# 模型现在可以决定是否调用工具
response = model_with_tools.invoke(message)

# 检查模型是否想调用工具
print("Tool calls:", response.tool_calls)

模型会返回一次**工具调用**请求，包含：
- `name`：要调用的工具名
- `args`：传给工具的参数
- `id`：用于追踪的唯一标识

接下来执行工具并继续对话：


In [ ]:
from langchain.messages import ToolMessage

# 执行工具调用
if response.tool_calls:
    tool_call = response.tool_calls[0]

    # 调用实际工具
    if tool_call["name"] == "get_weather":
        result = get_weather.invoke(tool_call["args"])

    elif tool_call["name"] == "search_movies":
        result = search_movies.invoke(tool_call["args"])

    # 用结果创建 ToolMessage
    tool_message = ToolMessage(
        content=result,
        tool_call_id=tool_call["id"]
    )

    # 带着工具结果继续对话
    final_response = model_with_tools.invoke([
        HumanMessage(content=message),
        response,
        tool_message
    ])

    final_response.pretty_print()

### 要点：
- 工具是用 `@tool` 装饰的 Python 函数
- 良好的描述能帮助模型判断何时使用各工具
- 工具调用流程：模型请求工具 → 执行工具 → 返回结果 → 模型综合出最终回答


## 第 4 部分：用 `create_agent()` 构建第一个智能体

<img src="../../images/react_agent.png" style="width: auto; max-height: 500px; border-radius: 8px;">

手动编排「LLM 调用 → 工具调用」的固定序列既繁琐又僵硬。相反，我们可以用**智能体（agent）**运行如下循环：
1. 模型决定调用哪个工具（如有）
2. 执行工具
3. 结果返回给模型
4. 重复直到任务完成

LangChain 用 `create_agent()` 让这件事变得简单 —— **大约 10 行代码就能构建一个智能体！**
预构建智能体会处理上述循环，你只需指定系统提示（system prompt）与工具。


In [ ]:
from langchain.agents import create_agent

# 创建带工具的智能体
agent = create_agent(
    model=model,
    tools=[get_weather, search_movies],
    system_prompt="You are a helpful assistant that can check weather and recommend movies."
)

# 使用智能体
result = agent.invoke({
    "messages": [HumanMessage(content="What's the weather in NYC? (40.71° N, 74.01° W) Also recommend some sci-fi movies.")]
})

# 打印最终回复
for message in result["messages"]:
    message.pretty_print()

### 刚才发生了什么？

智能体自动完成了：
1. 分析用户请求
2. 调用 `get_weather("NYC")`
3. 调用 `search_movies("sci-fi")`
4. 把结果综合成自然回复

让我们可视化智能体结构：


In [ ]:
agent

### 要点：
- `create_agent()` 大约 10 行即可构建完整智能体
- 智能体自动处理「推理 → 行动 → 观察」循环
- 基于 LangGraph，具备生产级能力（持久化、流式、人机协同）


## 第 5 部分：添加记忆与状态

目前每次调用都是独立的。下面添加**记忆（memory）**，让智能体在多次交互中保持上下文。

LangGraph 使用**检查点（checkpointer）**来保存与恢复状态：


In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langsmith import uuid7

# 创建用于记忆的检查点（checkpointer）
checkpointer = MemorySaver()

# 创建带记忆的智能体
agent_with_memory = create_agent(
    model=model,
    tools=[get_weather, search_movies],
    system_prompt="You are a helpful assistant.",
    checkpointer=checkpointer
)

# 为本次对话创建线程（thread）
config = {"configurable": {"thread_id": uuid7()}}

# 第一次交互
result1 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "My name is Alice and I love sci-fi movies."}]},
    config=config
)

print("Response 1:", result1["messages"][-1].content)

# 第二次交互 —— 智能体能记住！
result2 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What's my name and what movies do I like?"}]},
    config=config
)
print("\nResponse 2:", result2["messages"][-1].content)

### 理解状态（State）与线程（Thread）

- **状态（State）**：智能体的「记忆」—— 包含消息历史与任意自定义数据
- **线程（Thread）**：由 `thread_id` 标识的一次会话
- **检查点（Checkpointer）**：每步后保存状态，从而支持记忆与错误恢复

每个线程彼此独立：


In [ ]:
# 新线程 —— 智能体不会记得 Alice
new_config_with_new_thread_id = {"configurable": {"thread_id": uuid7()}}

result3 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What's my name?"}]},
    config=new_config_with_new_thread_id
)
print("New thread response:", result3["messages"][-1].content)

### 要点：
- 检查点使跨交互记忆成为可能
- Thread ID 用来隔离不同对话
- 状态自动持久化 —— 无需手动管理状态！


## 第 6 部分：流式输出以提升体验

LLM 响应可能较慢。**流式（Streaming）** 可实时展示进度，显著改善用户体验。

LangChain 支持多种流式模式：


### 流式输出智能体步骤


In [ ]:
# 用 stream_mode="updates" 流式输出智能体进度
print("Streaming agent steps:\n")

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What's the weather in Boston? (42.36° N, 71.06° W) "}]},
    stream_mode="updates"
):
    for node_name, data in chunk.items():
        print(f"Step: {node_name}")
        if "messages" in data:
            message = data["messages"][-1]
            if hasattr(message, 'tool_calls') and message.tool_calls:
                print(f"   Tool call: {message.tool_calls[0]['name']}")
            elif hasattr(message, 'content'):
                print(f"   Content: {message.content[:100]}..." if len(message.content) > 100 else f"   Content: {message.content}")
        print()

### 流式输出 LLM Token

要获得类似 ChatGPT 的体验，可在生成时逐 token 流式输出：


In [ ]:
# 用 stream_mode="messages" 流式输出 token
print("Streaming tokens:\n")

for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "Tell me about LangGraph in one sentence."}]},
    stream_mode="messages"
):
    # 只打印来自 model 节点的内容
    if metadata.get('langgraph_node') == 'model':
        # 从 content blocks 中取出文本
        for block in token.content_blocks:
            if block.get('type') == 'text' and block.get('text'):
                print(block['text'], end='', flush=True)

print("\n")  # 末尾换行

### 要点：
- `stream_mode="updates"` —— 查看每个智能体步骤（便于调试）
- `stream_mode="messages"` —— 流式输出 LLM token（类 ChatGPT 体验）
- 流式是内置能力 —— 无需额外配置！


## 第 7 部分：综合运用 —— 一个更贴近实际的例子

让我们构建一个更贴近现实的智能体，综合前面所学：


In [ ]:
# 创建更贴近实际的工具
@tool
def get_user_preferences(user_id: str) -> str:
    """Get a user's saved preferences."""
    # 模拟用户数据库
    preferences = {
        "alice": "Loves sci-fi movies, prefers warm weather destinations",
        "bob": "Enjoys comedy films, likes cold climates for travel"
    }
    return preferences.get(user_id.lower(), "No preferences found")

@tool
def book_recommendation(genre: str, user_preferences: str = "") -> str:
    """Get personalized movie recommendations based on genre and user preferences."""
    recommendations = {
        "sci-fi": "Based on your preferences, try: Arrival, Ex Machina, or The Martian",
        "comedy": "Based on your preferences, try: The Big Lebowski, Anchorman, or Bridesmaids"
    }
    return recommendations.get(genre.lower(), "No recommendations available")

# 创建一个乐于助人的助手智能体
assistant = create_agent(
    model=model,
    tools=[get_weather, get_user_preferences, book_recommendation],
    system_prompt="""You are a helpful personal assistant.

    You can:
    - Check weather for any city
    - Look up user preferences
    - Recommend movies based on preferences

    Always be friendly and personalize your responses based on user preferences.""",
    checkpointer=MemorySaver()
)

# 演示对话
config = {"configurable": {"thread_id": uuid7()}}

print("=" * 50)
print("PERSONAL ASSISTANT DEMO")
print("=" * 50 + "\n")

# 交互 1
print("User: Hi, I'm Alice. Can you check my preferences and recommend a movie?\n")
result = assistant.invoke(
    {"messages": [{"role": "user", "content": "Hi, I'm Alice. Can you check my preferences and recommend a movie?"}]},
    config=config
)
print(f"Assistant: {result['messages'][-1].content}\n")

# 交互 2
print("User: Also, what's the weather like in San Francisco? (37.77° N, 122.42° W) \n")
result = assistant.invoke(
    {"messages": [{"role": "user", "content": "Also, what's the weather like in San Francisco? (37.77° N, 122.42° W) "}]},
    config=config
)
print(f"Assistant: {result['messages'][-1].content}\n")

print("=" * 50)

## 第 8 部分：下一步 —— 探索 LangGraph 原语

我们一直在用 `create_agent()`，它构建在 **LangGraph** 之上。LangGraph 通过三个核心原语，让你完全掌控智能体行为：

### LangGraph 核心概念：

1. **状态（State）** 
   - 在节点之间传递的共享数据结构
   - 代表智能体的「记忆」
   - 可包含消息、自定义数据等

2. **节点（Nodes）** 
   - 处理状态的 Python 函数
   - 每个节点完成一项具体任务
   - 例如：调用 LLM、执行工具、校验输入

3. **边（Edges）** 
   - 定义节点之间的流转
   - 可以是普通边（总是走向下一节点）
   - 也可以是条件边（按逻辑决定）


### 何时用 `create_agent()`，何时用自定义 LangGraph？

**适合用 `create_agent()` 的情况：**
- 构建标准 ReAct 风格智能体
- 需要快速原型
- 默认行为已满足需求

**适合用自定义 LangGraph 的情况：**
- 需要自定义控制流（例如审批工作流）
- 构建多智能体系统
- 实现人机协同（HITL）模式
- 有复杂状态管理需求

更高级的模式可参考：
- [LangGraph 文档](https://langchain-ai.github.io/langgraph/)
- [LangChain Academy](https://academy.langchain.com/)
- 本仓库中的 `multi_agent.ipynb`（LangGraph 201）


## 从零构建 ReAct 智能体


为理解 LangGraph 原语如何工作，我们重建上面那个智能体，但这次**不使用** `create_agent()`。


### 状态（State）


我们的聊天机器人很简单 —— 就是带一些工具的 LLM —— 因此状态里只需维护一份随对话增长的人类消息、AI 消息与工具消息列表。


In [ ]:
from typing_extensions import TypedDict
from typing import Annotated, List
from langgraph.graph.message import AnyMessage, add_messages
from langgraph.managed.is_last_step import RemainingSteps

class State(TypedDict):
    # 包含随对话增长的完整对话历史
    messages: Annotated[List[AnyMessage], add_messages]

### 节点（Nodes）

可以复用上面定义的工具。但我们需要一个节点来处理工具调用，并把结果格式化为 `ToolMessage` 供 LLM 使用。不必自己写节点，可使用 LangGraph 预构建的 `ToolNode`。


In [ ]:
from langgraph.prebuilt import ToolNode
# 节点
tools = [search_movies, get_weather]
tool_node = ToolNode(tools)

接下来创建主助手节点。


In [ ]:
def assistant(state: State):
    # 构造发给 LLM 的消息列表，以固定的系统提示开头
    system_prompt = "You are a helpful assistant that can check weather and recommend movies."
    all_messages = [SystemMessage(system_prompt)] + state["messages"]
    # 调用 LLM
    response = model_with_tools.invoke(all_messages)
    # 用 LLM 的回复更新状态
    return {"messages": [response]}

现在需要定义连接各节点的控制流，这就是**边（edges）**的概念。

**边是节点之间的连接，定义图的流转。**
* **普通边（Normal edges）** 是确定性的，总是从一节点到其指定目标
* **条件边（Conditional edges）** 用于在节点间动态路由，实现为根据逻辑返回下一节点的函数

本例中，我们需要从子智能体出发的**条件边**，以决定：
- 调用工具，或
- 若用户查询已完成则路由到结束


In [ ]:
# 决定是否继续的条件边
def should_continue(state: State):
    messages = state["messages"]
    last_message = messages[-1]
    # LLM 想调用工具，应执行工具并继续
    if last_message.tool_calls:
        return "continue"
    # LLM 返回了回复而非工具调用，流程结束
    # 否则若有工具调用则继续
    else:
        return "end"

#### 编译图（Compile Graph）！

既已定义状态、节点与条件边，把它们组装起来，构建我们的 ReAct 智能体！


In [ ]:
from langgraph.graph import StateGraph, START, END

# 先用我们的 State 类定义 builder
builder = StateGraph(State)

# 添加节点
builder.add_node("assistant", assistant)
builder.add_node("tool_node", tool_node)


# 添加边
# 首先定义起始节点。查询总是先路由到 assistant 节点。
builder.add_edge(START, "assistant")

# 在调用 LLM 之后加入条件边
builder.add_conditional_edges(
    "assistant",
    should_continue,
    {
        # 若为 `continue`，则调用工具节点。
        "continue": "tool_node",
        # 否则结束。
        "end": END,
    },
)

# 调用工具后总是回到 LLM
builder.add_edge("tool_node", "assistant")

agent = builder.compile(name="agent")
agent

我们已成功从零重建一个与用 `create_agent` 做出的基础版相同的 ReAct 智能体。试一试！


In [ ]:
from langchain.messages import HumanMessage

question = "What is the weather in SF today (37.77° N, 122.42° W), and what are some good Sci-Fi movies?"

result = agent.invoke({"messages": HumanMessage(content=question)})

for message in result["messages"]:
    message.pretty_print()

## 🎉 恭喜！

你已掌握用 LangChain 与 LangGraph 构建智能体的核心概念：

 **模型（Models）** —— 跨供应商的标准化接口  
 **消息（Messages）** —— 对话的构建块  
 **工具（Tools）** —— 扩展 LLM 能力  
 **智能体（Agents）** —— 自动化推理与行动循环  
 **记忆（Memory）** —— 跨交互保持上下文  
 **流式（Streaming）** —— 实时用户体验  
 **LangGraph** —— 支撑这一切的基础

### 接下来做什么？

1. **用你自己的工具与场景构建智能体**
2. **在 `multi_agent.ipynb` 中探索高级模式**
3. **用 [LangSmith](https://smith.langchain.com) 做调试**
4. **借助 LangGraph 的持久化与错误恢复部署到生产**

### 资源：

- [LangChain 文档](https://python.langchain.com/docs/introduction/)
- [LangGraph 文档](https://langchain-ai.github.io/langgraph/)
- [LangSmith 调试](https://smith.langchain.com)
- [LangChain Academy](https://academy.langchain.com/)
<br> 
<br> 
---
<br> 

**祝构建愉快！**
